In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from langchain.chat_models import init_chat_model
from typing import Callable

large_model = init_chat_model("google_genai:gemini-3.1-flash-lite")
standard_model = init_chat_model("google_genai:gemini-3.1-flash-lite")


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [3]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [4]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "Oh, I'm so glad you asked! Yes, I actually just finished doing that a few minutes ago. \n\nI checked the soil and it was looking a little dry, so I gave it a good drink. Hopefully, that helps it perk up a bit—it’s been looking a little droopy lately, hasn’t it? \n\nIs there anything else you need me to take care of around the office today?", 'extras': {'signature': 'EnMKcQFpFH0TOuIXwq1GHB/KiTOTehFjThjcZVfQYb2itM8oPmXfqsT6bC5PD/pkDENIgwElaZOCobCIXiWStFhdbwEnkoUpDMBhtlCUrWG6ooQZ6z8Yn50py66p1rSuCMT2HT/QcldKEWHMP4/eeHdxAyFe'}}]


In [5]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.1-flash-lite


In [6]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

[{'type': 'text', 'text': "I’ve been keeping an eye on the roots, and I think we’re still okay for a while. \n\nThere’s currently about an inch of space between the soil and the rim, but I haven't seen any roots poking out of the drainage holes at the bottom yet. If we see any roots starting to peek through or if the soil starts drying out significantly faster than it does now, that'll be our signal to move it up a size.\n\nI'll keep checking it every time I water, though—should I put a note in my calendar to re-assess the root health in about a month?", 'extras': {'signature': 'EnMKcQFpFH0TGVeBePadxHgGXdKtzbus2WAKYXd0GqhQlIHQOmRbUsz78wvrFLGeoudcXFVKXAlD8q1W6qPDOzGcWOoYIZvmEaKAp5HN0DjbvyQQi/J8yyg5dzlIsXbNW9duIv/OfMcfp0u3IcQk2kvKREyb'}}]


In [7]:
print(response["messages"][-1].response_metadata["model_name"])

gemini-3.1-flash-lite
